In [ ]:
import asyncio
import os

from dotenv import load_dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import Message, Content

load_dotenv(override=True)

In [ ]:
credential = AzureCliCredential()
client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"],
    credential=credential,
)

# Creating our agent
agent = client.as_agent(
    instructions="You are Batman, the Dark Knight of Gotham City.",
    name="Batman-Agent"
)

In [ ]:
# Running a chat loop with the agent
async def chat_with_agent():
    print("Chatting with the agent. Type 'exit' to quit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == 'exit':
            break
        response = await agent.run(user_input)
        print(f"Agent: {response.text}")

await chat_with_agent()

In [ ]:
# Running the agent with streaming responses
async def streaming_chat():
    async for update in agent.run("Tell me something interesting about Gotham City in 10 points", stream=True):
        if update.text:
            print(update.text, end="", flush=True)
    print()  # New line after streaming is complete

await streaming_chat()

In [ ]:
# Load image from local file
with open("../data/joker_interrogation_scene.png", "rb") as image_file:
    image_bytes = image_file.read()

# Create a message with text and image data
message = Message(
    role = "user",
    contents = [
        Content.from_text(text = "Tell me about the incident in this image."),
        Content.from_data(
            data = image_bytes,
            media_type = "image/png"
        )
    ]
)

In [ ]:
# Creating an async function for the entire image analysis run
async def analyze_image():
    response = await agent.run(message)
    print(f"Agent: {response.text}")

# invoke the async function
await analyze_image()